# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [ ]:
# ??? TOOL 1: Calculator

import re


def calculator(expression: str) -> str:
    """Evaluate a mathematical expression safely with a whitelist."""
    try:
        cleaned = expression.strip()
        cleaned = re.sub(r"^(calculate|compute|solve|what is)\s+", "", cleaned, flags=re.I)
        cleaned = cleaned.replace("^", "**")
        if not re.fullmatch(r"[0-9\.\+\-\*\/\%\(\)\s\*\*]+", cleaned):
            return "Error in calculation"
        result = eval(cleaned, {"__builtins__": {}}, {})
        return str(result)
    except Exception:
        return "Error in calculation"

In [ ]:
# ??? TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = re.findall(r"[A-Za-z][A-Za-z0-9_-]+", text.lower())
        stop_words = {"about", "from", "where", "which", "their", "there", "these", "those", "would", "could", "should", "might", "being", "have", "has", "had", "this", "that", "with", "into", "your", "what", "when", "will", "they", "them", "then", "than", "were", "been", "also", "only", "more", "most", "some", "such", "over", "under"}
        keywords = []
        seen = set()
        for word in words:
            if len(word) < 5 or word in stop_words or word in seen:
                continue
            seen.add(word)
            keywords.append(word)
            if len(keywords) == 5:
                break
        return keywords
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [ ]:
# ?? AGENT FUNCTION (TO IMPLEMENT)

import json
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(message)s")


def _general_response(query: str) -> str:
    return (
        "I can help with calculations, keyword extraction, and general questions. "
        f"You asked: {query}"
    )


def _detect_math_query(query_lower: str) -> bool:
    math_markers = ["calculate", "compute", "solve", "sum", "add", "subtract", "multiply", "divide", "+", "-", "*", "/", "%"]
    return any(marker in query_lower for marker in math_markers) and bool(re.search(r"\d", query_lower))


def _detect_keyword_query(query_lower: str) -> bool:
    return any(marker in query_lower for marker in ["keyword", "keywords", "extract keywords", "key words"])


def agent(query: str):
    query_lower = query.lower().strip()

    if not query_lower:
        return {
            "type": "error",
            "result": "Query cannot be empty"
        }

    try:
        if _detect_math_query(query_lower):
            expression = re.sub(r"^(calculate|compute|solve)\s+", "", query, flags=re.I)
            result = calculator(expression)
            if result == "Error in calculation":
                return {"type": "error", "result": result}
            return {
                "type": "calculation",
                "result": result
            }

        if _detect_keyword_query(query_lower):
            text = re.sub(r".*?(?:keywords?|key words?)\s*(?:from)?\s*", "", query, flags=re.I)
            if not text.strip():
                text = query
            keywords = extract_keywords(text)
            return {
                "type": "keywords",
                "result": keywords
            }

        return {
            "type": "general",
            "result": _general_response(query)
        }

    except Exception as exc:
        logging.exception("Agent failed")
        return {
            "type": "error",
            "result": f"Unexpected error: {exc}"
        }


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [ ]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))